# 1. Header, Citations, Purpose & Topic Bridges

**Topic:** Neural Operators: Fourier Neural Operator (FNO), DeepONet, and MeshGraphNets
**Category:** 06_smooth_and_physics_informed_ai

### Citations & Core References
- Lu, L., Jin, P., & Karniadakis, G. E. (2019). *DeepONet: Learning Nonlinear Operators for Identifying Differential Equations Based on the Universal Approximation Theorem of Operators*. Nature Machine Intelligence.
- Li, Z., Kovachki, N., Azizzadenesheli, K., Liu, B., Bhattacharya, K., Stuart, A., & Anandkumar, A. (2020). *Fourier Neural Operator for Parametric Partial Differential Equations*. ICLR.
- Pfaff, T., Fortunato, M., Sanchez-Gonzalez, A., & Battaglia, P. W. (2020). *Learning Mesh-Based Simulation with Graph Networks*. ICLR.

## 2. Mathematical Foundations & Governing Equations

### Fourier Neural Operator (FNO-1D)
$$v_{l+1}(x) = \sigma\left( W v_l(x) + \mathcal{F}^{-1}\left( R(k) \cdot \mathcal{F}(v_l)(k) \right)(x) \right)$$
Truncating to $K$ lower Fourier modes guarantees mesh-independent resolution invariance in $\mathcal{O}(N \log N)$ complexity.

### DeepONet Universal Operator
$$G(u)(y) \approx \sum_{k=1}^p b_k(u(x_1), \dots, u(x_m)) \cdot t_k(y) + b_0 = \mathbf{b}(u)^T \mathbf{t}(y) + b_0$$
Decouples discrete sensor evaluations $u(x_i)$ (Branch) from continuous spatial queries $y$ (Trunk).

### MeshGraphNets Unstructured Graph Message Passing
$$e_{ij}' = \text{MLP}_e([v_i, v_j, e_{ij}]), \quad \bar{e}_i = \sum_{j \in \mathcal{N}(i)} e_{ij}', \quad v_i' = \text{MLP}_v([v_i, \bar{e}_i])$$

In [ ]:
# 3. Deterministic Setup & Micro-Data Generator
import math
import torch
import torch.nn.functional as F
from safetensors.torch import load_file

torch.manual_seed(42)

# FNO setup: B=2, d_in=1, d_v=4, S=8, K=4
B, d_v, S, K = 2, 4, 8, 4
fno_input = torch.randn(B, 1, S)
w_p = torch.randn(d_v, 1) * 0.5
b_p = torch.randn(d_v, 1) * 0.1
r_real = torch.randn(K, d_v, d_v) * 0.2
r_imag = torch.randn(K, d_v, d_v) * 0.2
w_bypass = torch.randn(d_v, d_v) * 0.2
w_q = torch.randn(1, d_v) * 0.5
b_q = torch.randn(1, 1) * 0.1

# DeepONet setup: m=8 sensors, p=8 latent features, 4 continuous query points
u_sensor = torch.randn(B, 8)
y_query = torch.linspace(0.0, 1.0, 4).unsqueeze(1)
w_b = torch.randn(8, 8) * 0.3
b_b = torch.randn(8) * 0.1
w_t = torch.randn(1, 8) * 0.3
b_t = torch.randn(8) * 0.1
b_out0 = torch.randn(1) * 0.1

print(f"FNO input: {fno_input.shape}, DeepONet sensor input: {u_sensor.shape}, queries: {y_query.shape}")

In [ ]:
# 4. First-Principles Implementation & Native Parity
# 1. FNO Forward Pass
v0 = F.gelu(w_p @ fno_input + b_p, approximate="none")

# Spectral Convolution via exact discrete Fourier matrix
k_idx = torch.arange(K).unsqueeze(1)
n_idx = torch.arange(S).unsqueeze(0)
angle = 2.0 * math.pi * k_idx * n_idx / S
c_mat = torch.cos(angle)
s_mat = -torch.sin(angle)

v_r = torch.matmul(v0, c_mat.t())
v_i = torch.matmul(v0, s_mat.t())
u_r = torch.einsum("bck, koc -> bok", v_r, r_real) - torch.einsum("bck, koc -> bok", v_i, r_imag)
u_i = torch.einsum("bck, koc -> bok", v_r, r_imag) + torch.einsum("bck, koc -> bok", v_i, r_real)

weights_k = torch.tensor([1.0] + [2.0] * (K - 1)).unsqueeze(1)
m_cos = torch.cos(angle) * weights_k
m_sin = torch.sin(angle) * weights_k
k_v0 = (1.0 / S) * (torch.matmul(u_r, m_cos) - torch.matmul(u_i, m_sin))

bypass = torch.einsum("oc, bcs -> bos", w_bypass, v0)
v1 = F.gelu(k_v0 + bypass, approximate="none")
fno_output = w_q @ v1 + b_q

# 2. DeepONet Forward Pass
branch_feat = F.gelu(u_sensor @ w_b + b_b, approximate="none")
trunk_feat = F.gelu(y_query @ w_t + b_t, approximate="none")
deeponet_output = branch_feat @ trunk_feat.t() + b_out0

print(f"FNO output: {fno_output.shape}, DeepONet output: {deeponet_output.shape}")

In [ ]:
# 5. Quantitative Invariants & Parity Verification
# Invariant 1: Spectral convolution equivalence to FFT
v0_fft = torch.fft.rfft(v0, n=S, dim=-1)[..., :K]
R_c = torch.complex(r_real, r_imag)
fft_spectral = torch.einsum("bck, koc -> bok", v0_fft, R_c)
fft_pad = torch.zeros(B, d_v, S // 2 + 1, dtype=torch.complex64)
fft_pad[..., :K] = fft_spectral
k_v0_fft = torch.fft.irfft(fft_pad, n=S, dim=-1)
diff_fft = torch.max(torch.abs(k_v0 - k_v0_fft)).item()
assert diff_fft < 1e-5, f"Discrete spectral conv FFT parity failed: {diff_fft:.2e}"

# Invariant 2: DeepONet batch linearity in branch features
b_double = torch.cat([branch_feat, branch_feat], dim=0)
out_double = b_double @ trunk_feat.t() + b_out0
assert torch.allclose(out_double[:B], deeponet_output), "DeepONet batch invariance failed"
print(f"All mathematical invariants (FFT spectral parity diff {diff_fft:.2e}, branch batch linearity) verified.")

In [ ]:
# 6. Performance Profile & Diagnostic Visualizations
import time

iters = 100
t0 = time.perf_counter()
for _ in range(iters):
    _ = branch_feat @ trunk_feat.t() + b_out0
dt = (time.perf_counter() - t0) / iters * 1000.0
print(f"DeepONet query evaluation latency: {dt:.3f} ms / step")

In [ ]:
# 7. Pathological Misuse vs. Corrected Implementation
# Pathological: Non-periodic signal evaluated with FNO without padding experiences Gibbs ringing
x_step = torch.tensor([0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0]).view(1, 1, 8)
x_fft = torch.fft.rfft(x_step, n=8, dim=-1)
x_trunc = torch.zeros_like(x_fft)
x_trunc[..., :2] = x_fft[..., :2] # severe truncation
x_ringing = torch.fft.irfft(x_trunc, n=8, dim=-1)
gibbs_error = torch.max(torch.abs(x_ringing - x_step)).item()
print(f"Pathological unpadded step function Gibbs ringing error: {gibbs_error:.4f}")

# Corrected: Smooth periodic continuation or domain boundary padding eliminates ringing artifacts
x_smooth = torch.sin(math.pi * torch.linspace(0.0, 1.0, 8)).view(1, 1, 8)
x_smooth_fft = torch.fft.rfft(x_smooth, n=8, dim=-1)
x_smooth_trunc = torch.zeros_like(x_smooth_fft)
x_smooth_trunc[..., :4] = x_smooth_fft[..., :4]
x_smooth_recon = torch.fft.irfft(x_smooth_trunc, n=8, dim=-1)
smooth_error = torch.max(torch.abs(x_smooth_recon - x_smooth)).item()
print(f"Corrected smooth periodic field reconstruction error: {smooth_error:.4f}")

## 8. Algorithmic Takeaways & Cross-Domain Isomorphisms

1. **Mesh-Independent Operator Learning**: FNO and DeepONet parameterize continuous infinite-dimensional operators rather than grid-bound vectors, enabling zero-shot super-resolution.
2. **Global Receptive Fields**: Spectral convolution in Fourier space acts on global domain modes simultaneously with $\mathcal{O}(N \log N)$ complexity.
3. **Unstructured Mesh Flexibility**: MeshGraphNets bridge operator learning to arbitrary finite-element meshes via message passing.